# Deep Learning Framework for Environmental Sound Recognition Using PyTorch
### Notebook 03 — CO4 Literature Comparison · CO5 Ablation · Faculty Submission

**Dataset:** UrbanSound8K · **Backbone:** official PANNs CNN14 pretrained on AudioSet · **Learning task:** single-label ten-class classification.

**Integrity:** This notebook analyzes **actual** Stage 4 results (3 full-fold runs) and one Stage 5 held-out test uploaded on 30 October 2026. It contains no invented run histories, accuracy scores, hyperparameter sweeps, or new experiments. It is a companion to Notebooks 01 and 02 and does not overwrite them.

**Read first:** the official faculty guidance in `project_submission_instructions` requires CO3 implementation, CO4 original-paper comparison, CO5 justified experimental contribution, a clear README, exact dependency versions, reproducible source, and **RegistrationNumber.zip**.

**The dataset authors recommend ten-fold evaluation. Our current result covers one test fold only**, so this notebook explicitly distinguishes our score from published multi-fold results.

## 1 · Safe notebook controls

`Restart & Run All` is safe by default: it only reads the **small evidence copies bundled with this notebook** and displays analysis. It does not install, train, re-test, move audio, delete caches, or create the final submission. Enable one action at a time once you have verified the evidence.

In [ ]:
from pathlib import Path
import json, sys, subprocess, csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

# FILL WITH YOUR OWN REAL DETAILS BEFORE FINAL PACKAGE (never share made-up details).
STUDENT_NAME = ""
REGISTRATION_NUMBER = ""
SECTION = ""

RUN_FACULTY_AUDIT = False
UPDATE_ROOT_README = False
CREATE_FINAL_SUBMISSION_ZIP = False
CLEAR_PYTHON_CACHES = False

HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / 'reports' / 'faculty_evidence').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Open project folder in VS Code and extract Stage 6–8 kit into its root before executing.')
EVIDENCE = ROOT / 'reports' / 'faculty_evidence'
print('Project root:', ROOT)
print('Evidence folder:', EVIDENCE)
print('Student details set:', bool(STUDENT_NAME.strip() and REGISTRATION_NUMBER.strip() and SECTION.strip()))


## 2 · Evidence provenance and single-held-out-fold protocol

The selected model was **`block6`**, chosen by *validation fold 2* after comparing 3 freezing-depth configurations. Test fold 1 was evaluated once afterward. The selected checkpoint was the validation-selected **epoch 4**, not the final epoch 6.

The source ID caveat is retained: five UrbanSound8K Freesound `fsID` values span official folds; disjoint *clips* do not imply full source-recording independence.

In [ ]:
ev = json.loads((EVIDENCE / 'heldout' / 'evaluation_summary.json').read_text(encoding='utf-8'))
assert ev['test_evaluated'] is True
assert ev['config'] == 'block6' and ev['test_fold'] == 1 and ev['validation_fold'] == 2
assert ev['test_samples'] == 873 and ev['checkpoint_epoch'] == 4
assert ev['stage4_best_checkpoint_sha256'] == '7c0e01a0078c225a18f4dc5eaa5110d051ef2c8effec7bcebb4e856e6659328c'
summary_fields = ['accuracy','macro_precision','macro_recall','macro_f1','macro_specificity_ovr','macro_roc_auc_ovr','test_cross_entropy']
display(pd.DataFrame({'metric':summary_fields, 'value':[ev[k] for k in summary_fields]}))
print('Test samples:', ev['test_samples'], 'Fold:', ev['test_fold'], 'Best checkpoint epoch:', ev['checkpoint_epoch'])
print('Note: NO 10-fold benchmark has been performed.')


## 3 · CO3 — Held-out performance and class-wise error analysis

In the confusion matrix, **rows are true classes**, **columns are predicted classes**. The matrix counts are derived from the real Stage 5 evaluation. OvR ROC-AUC is computed from *softmax probabilities*, not class predictions.

In [ ]:
per_class = pd.read_csv(EVIDENCE / 'heldout' / 'per_class_metrics.csv').sort_values('class_id')
assert per_class['class_id'].tolist() == list(range(10))
assert int(per_class['support'].sum()) == 873
assert int(per_class['tp'].sum()) == 773
assert np.isclose(float(np.mean(per_class['f1'])), ev['macro_f1'])
display(per_class[['class_id','class_name','support','precision','recall','f1','specificity_ovr','roc_auc_ovr']].round(4))
print('Correct:',int(per_class['tp'].sum()),' Errors:',873-int(per_class['tp'].sum()))


In [ ]:
display(Image(filename=str(EVIDENCE / 'heldout' / 'confusion_matrix.png'), width=800))

In [ ]:
fig, ax = plt.subplots(figsize=(10,4.8))
ax.bar(per_class['class_name'], per_class['recall'])
ax.set_xticks(range(10),per_class['class_name'],rotation=55,ha='right')
ax.set_ylim(0,1.05);ax.set_title('Held-out fold 1 — recall by official class');ax.set_ylabel('Recall')
ax.grid(axis='y',alpha=.2);plt.tight_layout();plt.show()

**Interpretation:** air-conditioner recall is **69%**, drilling **68%**, engine idling **82.29%**, and jackhammer **94.17%**. Mechanical classes have the most confusion. Conversely, **car horn, gunshot and siren each have 100% recall** on their respective held-out support counts (36, 35 and 86). This does **not** guarantee perfect performance on future data.

## 4 · CO5 — Controlled ablation of CNN14 fine-tuning depth

All three runs used the **same 8/1/1 folds, seed 67, six-epoch budget, waveform preprocessing, optimizer policy, BF16, batch size 1 and accumulation 4**. The chosen variable was **which pretrained layers were trainable**. This is a controlled comparison on *one validation fold*, not an invented new neural architecture.

In [ ]:
rows=[]; histories={}
for method in ('head','fc1','block6'):
    p=EVIDENCE/'training_runs'/method
    c=json.loads((p/'config.json').read_text(encoding='utf-8'))
    s=json.loads((p/'summary.json').read_text(encoding='utf-8'))
    h=pd.read_csv(p/'history.csv');histories[method]=h
    assert c['config']==method and c['diagnostic_only'] is False and c['test_fold']==1
    assert c['validation_fold']==2 and c['seed']==67 and c['max_train_batches']==0
    assert c['max_val_batches']==0 and s['best_epoch']==4 and s['test_evaluated'] is False
    rows.append({'config':method,'trainable_params':c['parameter_counts']['trainable'],
      'total_params':c['parameter_counts']['total'],'val_accuracy_%':100*s['best_val_accuracy'],
      'val_macro_F1':s['best_val_macro_f1'],'best_epoch':s['best_epoch'],
      'total_minutes':h['seconds'].sum()/60,
      'peak_cuda_MiB':h['peak_cuda_allocated_mib'].max(),
      'LR_head':c['lr_head'],'LR_backbone':c['lr_backbone']})
comparison=pd.DataFrame(rows)
display(comparison.round(4))
print('block6 vs head: +%.2f validation accuracy percentage points, +%.4f macro-F1' %
      (comparison.loc[2,'val_accuracy_%']-comparison.loc[0,'val_accuracy_%'],
       comparison.loc[2,'val_macro_F1']-comparison.loc[0,'val_macro_F1']))

In [ ]:
fig,ax=plt.subplots(figsize=(8,4.5))
for method,h in histories.items():
    ax.plot(h['epoch'],h['val_macro_f1'],marker='o',label=method)
ax.set(xlabel='Epoch',ylabel='Validation macro-F1',title='Controlled freezing-depth comparison (fold 2 validation)')
ax.grid(alpha=.2);ax.legend();plt.tight_layout();plt.show()

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
for method,h in histories.items():
    axes[0].plot(h['epoch'],h['train_loss'],label=method+' train')
    axes[0].plot(h['epoch'],h['val_loss'],ls='--',label=method+' validation')
    axes[1].plot(h['epoch'],h['val_accuracy']*100,marker='o',label=method)
axes[0].set(xlabel='Epoch',ylabel='Cross-entropy loss',title='Train and validation loss');axes[0].grid(alpha=.2)
axes[1].set(xlabel='Epoch',ylabel='Validation accuracy (%)',title='Validation accuracy');axes[1].grid(alpha=.2)
axes[0].legend(fontsize=8,ncol=2);axes[1].legend();plt.tight_layout();plt.show()

### Ablation interpretation and bias–variance considerations

`block6` improves validation macro-F1 **0.9010 versus 0.8638** (head-only), but uses **60,848,138 trainable parameters** compared with **20,490** for head-only. Its six-epoch training time increases **19.80 → 28.65 minutes**. This demonstrates a quality–compute trade-off, not parameter efficiency. Validation macro-F1 falls after epoch 4 as training progresses; therefore we report the epoch-4 checkpoint. Training-time augmentation/dropout affects online training accuracy, so do not treat raw train/validation accuracy differences as an exact generalization-gap statistic without considering train/eval modes.

**Original contribution:** *controlled study of where to unfreeze pretrained CNN14 for an environmental sound classification task*. Not a new CNN, novel optimizer, or new dataset.

## 5 · CO4 — Published UrbanSound8K methods (original papers)

**Important:** compare research context, NOT exact ranking. The published rows summarize their own protocols and usually multiple folds, while our row uses only held-out fold 1.

Original papers:

- Su et al. (2019), TSCNN-DS, https://doi.org/10.3390/s19071733
- Ting et al. (2021), Inception–Dense Blocks, https://doi.org/10.3390/s21165406
- Fang et al. (2022), RACNN, https://doi.org/10.1038/s41598-022-10382-x
- UrbanSound8K official instructions: https://urbansounddataset.weebly.com/urbansound8k.html

See `docs/CO4_PUBLISHED_METHODS.md` for detailed paper methods, external pretraining discussion, incomplete metric fields (NR), limitations and references.

In [ ]:
literature=pd.DataFrame([
 {'published_method':'TSCNN-DS (Su et al.)','year':2019,'input':'Multi-feature two-stream CNN + fusion','accuracy_%':97.20,'protocol':'Reported 10 experiments; paper N=10','parameters':'NR in primary comparison'},
 {'published_method':'Inception–Dense (Ting et al.)','year':2021,'input':'16 kHz log-Mel; lightweight CNN','accuracy_%':83.03,'protocol':'10 predefined folds; mean','parameters':'97,214'},
 {'published_method':'RACNN (Fang et al.)','year':2022,'input':'Log-Mel; resource-adaptive CNN','accuracy_%':97.51,'protocol':'Paper multi-fold mean ±0.18','parameters':'145,700'},
])
display(literature)
display(pd.DataFrame([{'our_method':'AudioSet CNN14 (block6)','year':2026,'accuracy_%':100*ev['accuracy'],'protocol':'ONE held-out fold 1 ONLY','total_params':81857561}]))
print('NO FAIR RANKING: different protocols, feature representations, pretraining and training resources.')

## 6 · Faculty mandatory README and evidence audit

The faculty requires: README with student name, registration and section; dataset source, distribution and preprocessing; architecture; **actual** optimizer/LR/batch/epochs/loss/weight decay and tuning explanation; learning curves; confusion matrix and metrics; original-paper CO4; quantitative CO5 ablation; source, notebook, dependency versions, and correctly named ZIP.

`tools/faculty_submission.py audit` only checks that the files and metadata exist and agree. **It does not certify educational novelty, reproducibility on a fresh PC, or ten-fold generalization.**

In [ ]:
if RUN_FACULTY_AUDIT:
    r=subprocess.run([sys.executable,str(ROOT/'tools'/'faculty_submission.py'),'audit','--project',str(ROOT)],
                     cwd=str(ROOT),capture_output=True,text=True)
    print(r.stdout);print(r.stderr);print('Exit:',r.returncode)
else: print('Faculty file audit disabled. Set RUN_FACULTY_AUDIT=True in Controls and re-run this cell.')

## 7 · Synchronize your existing README (optional, idempotent)

When enabled, this inserts or updates the managed faculty evidence section **without deleting other README content**. Keep original notes and README metadata; the final submission ZIP appends student details securely using values entered by you.

In [ ]:
if UPDATE_ROOT_README:
    sys.path.insert(0,str(ROOT))
    from tools.update_readme import update_readme
    print('Updated:',update_readme(ROOT))
else: print('README synchronization disabled; set UPDATE_ROOT_README=True and run once after reviewing results.')

## 8 · Optional Python cache cleanup

Cache cleaning is **not required** to restart the notebook. This is deliberately opt-in. It removes Python `__pycache__` directories and `*.pyc` files under `src/`, `tests/`, and `tools/` only. It **does not touch** data, models, full training histories, or notebook outputs. Restart the kernel after cleanup.

In [ ]:
if CLEAR_PYTHON_CACHES:
    import shutil
    removed=[]
    for part in ('src','tests','tools'):
        target=ROOT/part
        if target.is_dir():
            for q in target.rglob('__pycache__'):
                if q.is_dir(): shutil.rmtree(q);removed.append(str(q))
            for q in target.rglob('*.pyc'):
                if q.is_file(): q.unlink();removed.append(str(q))
    print('Removed',len(removed),'cache locations/files; dataset/checkpoints untouched.')
else: print('Cache cleanup disabled (recommended).')

## 9 · Create the exact faculty submission ZIP (opt-in)

Before enabling: enter **your true** student name, registration number (digits only), and section in Controls. Verify every notebook runs and displays correct evidence; verify `README.md` content, original citations and `requirements.lock.txt`; preserve your original `best.pt` file. The packager verifies the **selected best.pt SHA-256** before placing it in the archive, along with source/notebooks/evidence. The pretrained 1.4-GB AudioSet backbone and UrbanSound8K WAVs are excluded; `models/CHECKPOINT_ACCESS.md` explains how to acquire them. **The archive name is only your registration number**.

In [ ]:
if CREATE_FINAL_SUBMISSION_ZIP:
    assert STUDENT_NAME.strip() and REGISTRATION_NUMBER.isdigit() and SECTION.strip(), 'Fill student metadata first'
    cmd=[sys.executable,str(ROOT/'tools'/'faculty_submission.py'),'package','--project',str(ROOT),
         '--registration',REGISTRATION_NUMBER,'--student-name',STUDENT_NAME,'--section',SECTION]
    result=subprocess.run(cmd,cwd=str(ROOT),capture_output=True,text=True)
    print(result.stdout);print(result.stderr)
    if result.returncode: raise RuntimeError('Faculty ZIP blocked. Fix the audit errors; do not submit a partial ZIP.')
else: print('Packaging disabled. Set CREATE_FINAL_SUBMISSION_ZIP=True only after final checklist.')

## 10 · Faculty grading checklist and viva guide

| Teacher requirement | Actual evidence | Status |
|---|---|---|
| CO3 dataset/class/fold statistics | 8,732 metadata/WAVs; 10 classes/folds; full audit and notebook 01 | Verified |
| CO3 architecture/pretraining/preprocessing | Official PANNs CNN14 32 kHz; strict backbone load; mono 4 s waveform | Verified |
| CO3 genuine fine-tuning and training | Verified convolution gradients/updates; all three six-epoch training runs | Verified |
| CO3 evaluation/plots | Selected `block6` fold-1 test 88.55%, macro F1 0.8964, confusion matrix and per-class metrics | Verified |
| CO4 original papers | Three original UrbanSound8K papers with metric/protocol caveats | Prepared; present clearly |
| CO5 controlled ablation | Head / fc1 / block6 validation comparison, parameter and compute trade-offs | Verified, single split |
| Hyperparameter tuning | Actual single-seed/layer-depth study; settings held fixed, not grid searched | Documented |
| Multiple test folds | Authors recommend all 10; only fold 1 evaluated | Not completed; disclose |
| Submission notebook/README/source/lockfile | Verify on student's local project; execute audit | Pending local audit |
| Named final ZIP with real student details | Requires user-provided name, registration, section and selected checkpoint | Pending |

**Viva questions to prepare:** Why CNN14 and AudioSet? Why waveform rather than external log-Mel? How does partial fine-tuning differ from head-only? Why freeze BatchNorm stats? Why CrossEntropyLoss rather than BCE? Why test selection on validation macro-F1? Why can published ten-fold means not be compared as a single score? Why are some mechanical classes confused? Why does a zero gradient on one silent example not negate updates on other examples? What is the computational cost of unfreezing conv_block6?

**Final rule:** Do not report test results for head/fc1, ten-fold means, new hyperparameter sweeps, or new innovations that were never executed. Save this notebook (`Ctrl+S`) after checking its outputs.